# Ride-Sharing Follow-Up Survey: Supplementary Analysis

This notebook analyzes `ride-share-survey.xlsx`, a **second, separate** data-collection
instrument fielded after the main survey (`survey.csv`, N=199) closed. It covers
**ride-sharing pricing only** (no beauty-filter or LLM items), was administered as its
own Google Form, and reached 47 respondents.

**Read this before trusting any number below:**

- **Different consent model.** This form collected full name (required), email
  (optional), and a signature (optional) -- the main survey collected none of these
  by design. Those three columns are dropped **immediately on load**, in the very
  first code cell, before anything else touches the dataframe. No name, email, or
  signature appears anywhere else in this notebook.
- **Not verifiably independent of the main sample.** Because the main survey collected
  no identifying information, there's no way to check whether any of these 47 people
  also completed the original survey. This sample is therefore treated throughout as a
  **separate, supplementary sample** -- never silently pooled into the primary N=199 --
  with a clearly labeled pooled-sensitivity check reported separately where useful.
- **A demographically distinct population.** 100% male, skewing markedly older, and
  about 92% rural/small-town -- close to the demographic *complement* of the main
  study's own stated sampling-skew limitation (which underrepresents rural and older
  respondents). That makes this a genuine, if small and non-probability, check on
  whether the main study's findings hold outside its usual respondent profile.
- **Mostly identical wording.** 12 of 14 ride-sharing items are worded identically to
  the main survey; 2 items add concrete local examples (neighborhood names, and
  clarifying examples for a discount question) not present in the original wording.

All of the actual computation here is delegated to `run_followup_analysis.py` in this
same directory, which is the single source of truth for every number (mirroring how
`FairnessPerception.ipynb` wraps `run_analysis.py` for the main dataset). This notebook
re-runs that pipeline and displays/interprets its output.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 80)

## 1. Load and immediately de-identify

`followup_codebook.py` lists the 3 personally-identifying column positions
(`PII_COLS = [1, 2, 3]`: full name, email, signature). They are dropped in the same
line that loads the file -- there is no intermediate step where a cell could
accidentally display them.

In [2]:
from followup_codebook import COLS, PII_COLS

raw = pd.read_excel("../ride-share-survey.xlsx", sheet_name="Form Responses 1")
pii_names = [raw.columns[i] for i in PII_COLS]
print("Dropping these columns immediately (never used again):", pii_names)

df = raw.drop(columns=pii_names)
print(f"\nShape after de-identification: {df.shape}")
print(f"Timestamp range: {df[df.columns[0]].min()} to {df[df.columns[0]].max()}")

Dropping these columns immediately (never used again): ['YOUR FULL NAME IN BLOCK LATTER', 'Your Email please', 'Your Signature']

Shape after de-identification: (47, 19)
Timestamp range: 2025-08-24 11:49:50.946000 to 2025-12-10 20:34:37.855000


## 2. Run the analysis pipeline

`run_followup_analysis.py` reuses the same statistical machinery as the main
pipeline (`paired_comparison`, `min_detectable_effect`, `short_cat`, all imported
directly from `run_analysis.py`), so results here are computed the identical way --
Shapiro-Wilk normality check first, Wilcoxon signed-rank as the primary test when
normality fails, Cohen's $d$ and $\eta^2$ alongside, and a minimum-detectable-effect
sensitivity note.

In [3]:
import subprocess
result = subprocess.run([sys.executable, "run_followup_analysis.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("run_followup_analysis.py failed")

with open("followup_results.json", encoding="utf-8") as f:
    results = json.load(f)

print(json.dumps(results["meta"], indent=2, ensure_ascii=False))

  saved figure_followup_emergency_casual.png + figure_followup_emergency_casual.pdf

Wrote C:\Users\Admin\Documents\GitHub\Prompt-Manager\0. Paper Review - Before Submission\Perceived Fairness Project\analysis\followup_results.json

{
  "source_file": "ride-share-survey.xlsx",
  "n": 47,
  "timestamp_min": "2025-08-24 11:49:50.946000",
  "timestamp_max": "2025-12-10 20:34:37.855000",
  "pii_columns_dropped": [
    "full name",
    "email",
    "signature"
  ],
  "note": "Supplementary ride-sharing-only follow-up sample; analyzed separately from the main N=199 sample throughout (see module docstring for why). No beauty-filter or LLM items in this instrument."
}


## 3. Who answered this follow-up survey?

This is the headline context for everything below: this sample looks almost nothing
like the main study's sample.

In [4]:
demo = results["demographics"]

print("Age:", demo["age_pct"])
print("Gender:", demo["gender"], "(main sample was 65.8% male, 34.2% female)")
print("Residence:", demo["residence_pct"], "(main sample was 64.3% Major City)")
print(f"Prior algorithmic awareness: {demo['algo_awareness_pct_yes']}% Yes "
      f"(main sample: 66.3%)")

Age: {'27-30': 40.4, '24-26': 31.9, 'Above 30': 23.4, '21-23': 4.3}
Gender: {'Male': 47} (main sample was 65.8% male, 34.2% female)
Residence: {'Rural Area / Village': 48.9, 'Town or Small City': 42.6, 'Major City': 8.5} (main sample was 64.3% Major City)
Prior algorithmic awareness: 55.3% Yes (main sample: 66.3%)


**Takeaway.** This is a 100% male, disproportionately rural/small-town, older-skewing
sample -- close to the demographic complement of the main study's own stated
limitation ("the sample skews toward educated, urban, comparatively young
respondents... underrepresents rural and older perspectives"). No socioeconomic-status
item was asked in this instrument.

## 4. Does the flagship "exploitation framework" finding replicate?

The main study's headline ride-sharing finding is that an identical 20% price
increase is rated significantly less fair in a medical-emergency context than in a
casual context ($M=2.00$ vs. $M=2.17$, Wilcoxon $p=.006$, Cohen's $d=-0.19$). Here is
the same paired comparison, run on the follow-up sample.

In [5]:
main_rs = results["ride_share_emergency_vs_casual_main_sample_for_reference"]
fu_rs = results["ride_share_emergency_vs_casual_followup"]

comparison = pd.DataFrame({
    "Main sample": [main_rs["n"], main_rs["emergency_mean"], main_rs["casual_mean"],
                     main_rs["wilcoxon_p"], main_rs["cohens_d"], None],
    "Follow-up sample": [fu_rs["n"], fu_rs["emergency_mean"], fu_rs["casual_mean"],
                          fu_rs["wilcoxon_p"], fu_rs["cohens_d"], fu_rs["mde_d_at_this_n"]],
}, index=["N", "Emergency mean", "Casual mean", "Wilcoxon p", "Cohen's d",
          "Min. detectable d at 80% power"])
display(comparison)

,Main sample,Follow-up sample
N,185.000000,47.000000
Emergency mean,2.000000,2.574000
Casual mean,2.173000,2.596000
Wilcoxon p,0.005774,0.636575
Cohen's d,-0.194000,-0.024000
Min. detectable d at 80% power,NaN,0.417000


**This does not clearly replicate on its own -- but the test is weak, not
contradictory.** In the follow-up sample the emergency/casual gap all but disappears
(2.574 vs. 2.596, $p=.64$). Taken at face value that looks like a failure to
replicate. But at $N=47$, this sample can only reliably detect effects of
$d\geq0.42$ at 80% power -- nearly *double* the effect size the main study actually
found ($d=-0.19$). An effect this small was never likely to show up as significant
in a sample this size. The honest reading is: **inconclusive, not disconfirmed.**

A secondary, descriptive look by residence category (not independently
significance-tested given the small per-group $n$) at least shows the *direction* is
consistent for the two larger subgroups:

In [6]:
by_res = pd.DataFrame(fu_rs["by_residence"]).T
by_res["gap (casual - emergency)"] = by_res["casual_mean"] - by_res["emergency_mean"]
display(by_res)

,n,emergency_mean,casual_mean,gap (casual - emergency)
Major City,4.0,2.250,1.250,-1.000
Rural Area / Village,23.0,2.826,2.957,0.131
Town or Small City,20.0,2.350,2.450,0.100


Rural (n=23) and Town/Small-City (n=20) respondents both rate the casual context as
*slightly* fairer than the emergency context -- the same direction as the main
finding, just far too small individually to be conclusive. The Major City subgroup
(n=4) reverses this, but a 4-person cell is not informative on its own.

As a final robustness check -- **not** a primary statistic, since the two samples
used different instruments, different consent models, and unverifiable
independence -- pooling both samples naively does not flip the finding:

In [7]:
pooled = results["ride_share_pooled_sensitivity_check"]
print(f"Pooled N={pooled['n']}: Wilcoxon p={pooled['wilcoxon_p']:.4f}, "
      f"Cohen's d={pooled['cohens_d']}")
print()
print(pooled["note"])

Pooled N=232: Wilcoxon p=0.0071, Cohen's d=-0.159

SENSITIVITY CHECK ONLY, not a primary statistic: pools the main N=199 sample with the 47-respondent follow-up sample despite their different instruments, consent models, and unverifiable independence. Reported only to show the direction/magnitude does not flip when naively pooled; the paper's primary figures are the two samples reported separately above.


## 5. Does the "universal transparency demand" finding replicate?

This is the more decisive result in this notebook. Two items ask essentially the same
question as the main survey's RQ4 items, in this ride-sharing-specific instrument.

In [8]:
cmp = results["awareness_and_attitude_vs_main_sample"]

rows = []
for key, label in [("aware_diff_pricing", "Aware apps show different prices"),
                    ("should_explain_fare", "Apps should explain fare calculation"),
                    ("transparency_fairer", "Transparency would make pricing feel fairer")]:
    c = cmp[key]
    rows.append([label, f"{c['followup_pct_yes']}% (n={c['followup_n']})",
                 f"{c['main_pct_yes']}% (n={c['main_n']})",
                 f"p={c['p']:.4f}" if c['p'] >= .001 else "p<.001",
                 c["min_expected_count"]])
display(pd.DataFrame(rows, columns=["Item", "Follow-up %", "Main sample %",
                                     "Chi-square p", "Min. expected cell count"]))

,Item,Follow-up %,Main sample %,Chi-square p,Min. expected cell count
0,Aware apps show different prices,91.5% (n=47),69.3% (n=199),p=0.0036,12.42
1,Apps should explain fare calculation,95.7% (n=47),86.4% (n=199),p=0.1262,5.54
2,Transparency would make pricing feel fairer,100.0% (n=47),94.0% (n=199),p=0.1771,2.29


**Transparency demand holds up.** 95.7% of this rural/older/male sample wants apps
to explain fare calculation (vs. 86.4% in the main sample) and 100% say transparency
would make pricing feel fairer (vs. 94.0%) -- neither difference is statistically
significant, meaning demand for transparency is *at least as high* in a population the
main study's own sample barely reached. This is a genuine, independent piece of support
for the paper's claim that transparency/consent/control read as a broadly shared,
demographically-unmoderated expectation rather than something concentrated in the
main study's more urban, educated respondent profile.

**Awareness of differential pricing is significantly *higher* here** (91.5% vs. 69.3%,
$p=.004$) -- worth noting on its own as a non-obvious finding, not just a footnote.

## 6. A genuine surprise: "is this pricing fair overall?"

This one item diverges sharply from the main sample, and it does not fit neatly into
either finding above. It deserves to be reported plainly rather than explained away.

In [9]:
fo = results["fair_overall_vs_main_sample"]
print(f"Follow-up sample: {fo['followup_pct_fair']}% say the pricing is fair overall (n={fo['followup_n']})")
print(f"Main sample:      {fo['main_pct_fair']}% say the pricing is fair overall (n={fo['main_n']})")
print(f"Chi-square p = {fo['p']:.2e}")

Follow-up sample: 61.7% say the pricing is fair overall (n=47)
Main sample:      26.1% say the pricing is fair overall (n=199)
Chi-square p = 6.97e-06


61.7% of the follow-up sample calls the pricing "fair overall," against 26.1% in the
main sample -- a large, highly significant gap in the *opposite* direction from what
the higher awareness result might suggest. More aware, in this sample, does not mean
more critical.

This notebook does not attempt to explain *why* -- doing so from this data alone would
be speculation dressed up as an interpretation. Plausible, untested candidates: a
different underlying relationship to ride-sharing pricing in this population (which may
be closer to the driver/operational side of the platform economy than the main
sample's university-affiliated respondents), a different survey administration context
(named consent suggests this may have been collected in person or through a more
mediated channel rather than the main survey's anonymous self-administered form), or a
genuine attitudinal difference by age, geography, or economic position that this dataset
cannot separate. This is flagged in the paper as an open question for future,
purpose-built research rather than resolved here.

## 7. Summary for the paper

| Main-study finding | What this follow-up sample shows |
|---|---|
| Emergency pricing rated less fair than casual (small effect) | Not significant here, but the sample is underpowered (MDE=0.42) to detect an effect this small; direction holds descriptively in the two larger residence subgroups; pooled check does not overturn the main finding |
| Transparency/consent demand is broad and demographically unmoderated | **Replicates, and strengthens the claim** -- equally high (or higher) demand for fare transparency in a demographically opposite sample |
| (not previously tested) | Awareness of differential pricing is significantly *higher* in this sample (91.5% vs. 69.3%) |
| (not previously tested) | "Fair overall" judgment is dramatically *higher* in this sample (61.7% vs. 26.1%) -- an open question, not resolved here |

Practically, for UI/UX design: the transparency-demand replication is the strongest
actionable signal here -- it says a transparency-first design default is not just an
urban/educated-user preference, and teams should not assume rural or older users care
less about fare explainability. The "fair overall" divergence is a caution against the
opposite assumption -- that all users will read a given pricing practice through the
same moral lens the main sample used; local calibration of any "this may be exploitative"
framing or intervention is likely necessary rather than a single message that travels
unchanged across contexts.